# 带时间窗与固定休息的容量约束车辆路径问题

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-time-windows-and-regular-breaks](https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-with-time-windows-and-regular-breaks)


## 问题描述

**在带时间窗与固定休息的容量约束车辆路径问题**中,一组具有相同容量的配送车辆必须为客户提供服务。客户具有已知的营业时间以及对单一商品的需求。车辆从一个共同的配送中心出发并返回,且必须为驾驶员安排固定休息。目标是最小化总延误、所用车辆数量以及总行驶距离。


### 学习要点

- 添加 list decision variables 以建模每辆卡车的客户序列
- 添加 integer decision variables 以建模两次休息之间的时间间隔
- 使用 recursive lambda function 定义数组,以计算客户的访问时间与驾驶员的休息开始时间
- 将延误建模为软约束(目标项而非硬约束)


## 数据

我们提供的带时间窗与固定休息的车辆路径问题算例来自 [Solomon 算例](http://web.cba.neu.edu/~msolomon/problems.htm)。数据文件的格式如下:

- 第一行给出算例的名称
- 第五行包含车辆数量及其公共容量
- 从第 10 行起,每个客户(从配送中心开始):

- 客户的索引
- x 坐标
- y 坐标
- 需求
- 最早到达时间
- 最晚到达时间
- 服务时间


## 建模方法

带时间窗与固定休息的容量约束车辆路径问题的 Hexaly 模型在 CVRPTW 模型的基础上扩展得到。关于该问题的路径与时间窗部分,我们请读者参阅该模型。

为了对此建模,我们引入了整型决策变量,表示每辆卡车相邻休息之间的时间间隔。通过以休息频率作为这些决策的上界,我们确保休息在整个规划时段内均匀分布。实际的休息时间则通过对这些间隔进行累积求和得到。

将休息纳入路径时间安排遵循以下原则:无论休息发生在行驶段、等待期还是服务期间,其固定时长都会被加到当前时间,从而使路径上的所有后续事件相应延后。

最后,目标与 CVRPTW 相同:我们按字典序依次最小化总延误、所用车辆数量以及总行驶距离。OptAgent 按目标声明顺序执行字典序优化,与 Hexaly 原实现保持一致。


## Python 实现


In [1]:
import math
from pathlib import Path

from optagent import ModelBuilder, solve

# Breaks parameters (15 minutes every 4 hours)
BREAKFREQUENCY = 60 * 4
BREAKDURATION = 15


def read_instance(filename):
    lines = Path(filename).read_text().splitlines()
    nb_trucks = int(lines[4].split()[0])
    truck_capacity = int(lines[4].split()[1])

    customer_lines = [line for line in lines[9:] if line.strip()]
    customers = []
    for line in customer_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        customers.append(
            {
                "id": int(parts[0]),
                "x": int(parts[1]),
                "y": int(parts[2]),
                "demand": int(parts[3]),
                "ready": int(parts[4]),
                "due": int(parts[5]) + int(parts[6]),
                "service": int(parts[6]),
            }
        )
    depot_x = customers[0]["x"]
    depot_y = customers[0]["y"]
    max_horizon = customers[0]["due"]
    customers = customers[1:]
    nb_customers = len(customers)

    demands = [c["demand"] for c in customers]
    earliest = [c["ready"] for c in customers]
    latest = [c["due"] for c in customers]
    service_time = [c["service"] for c in customers]

    dist_matrix = [
        [
            math.sqrt((customers[i]["x"] - customers[j]["x"]) ** 2 + (customers[i]["y"] - customers[j]["y"]) ** 2)
            for j in range(nb_customers)
        ]
        for i in range(nb_customers)
    ]
    dist_depot = [
        math.sqrt((depot_x - customers[i]["x"]) ** 2 + (depot_y - customers[i]["y"]) ** 2) for i in range(nb_customers)
    ]

    return {
        "nb_customers": nb_customers,
        "nb_trucks": nb_trucks,
        "truck_capacity": truck_capacity,
        "dist_matrix": dist_matrix,
        "dist_depot": dist_depot,
        "demands": demands,
        "earliest": earliest,
        "latest": latest,
        "service_time": service_time,
        "max_horizon": max_horizon,
    }


def build_cvrptwrb_model(data):
    nb_customers = data["nb_customers"]
    nb_trucks = data["nb_trucks"]
    truck_capacity = data["truck_capacity"]
    max_horizon = data["max_horizon"]

    nb_breaks = int(math.ceil(max_horizon / BREAKFREQUENCY) + 1)

    model = ModelBuilder()
    customers_sequences = [model.list(nb_customers, name=f"truck_{k}_customers") for k in range(nb_trucks)]
    model.constraint(model.partition(customers_sequences), name="partition")

    demands_array = model.array(data["demands"])
    earliest_array = model.array(data["earliest"])
    latest_array = model.array(data["latest"])
    service_time_array = model.array(data["service_time"])
    dist_matrix_array = model.array(data["dist_matrix"])
    dist_depot_array = model.array(data["dist_depot"])

    # Break gap decision variables and cumulative start times per truck.
    breaks_start_times = []
    for k in range(nb_trucks):
        gaps_k = [
            model.int(
                default=BREAKFREQUENCY,
                lb=1,
                ub=BREAKFREQUENCY,
                name=f"gap_{k}_{b}",
            )
            for b in range(nb_breaks)
        ]
        # Cumulative sum: each break_start = sum of prior gaps + offsets.
        truck_breaks = []
        cumulative = 0
        for b in range(nb_breaks):
            cumulative = cumulative + gaps_k[b]
            truck_breaks.append(cumulative + BREAKDURATION * b)
        breaks_start_times.append(model.array(truck_breaks))
        # Last break must extend past the planning horizon.
        model.constraint(
            model.at(breaks_start_times[k], nb_breaks - 1) >= max_horizon + 1,
            name=f"breaks_cover_{k}",
        )

    trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]

    dist_routes = []
    end_times = []
    for k in range(nb_trucks):
        sequence = customers_sequences[k]
        c = model.count(sequence)

        demand_lambda = model.lambda_function(lambda j: demands_array[j])
        route_quantity = model.sum(sequence, demand_lambda)
        model.constraint(route_quantity <= truck_capacity, name=f"cap_{k}")

        dist_lambda = model.lambda_function(
            lambda i: dist_matrix_array[sequence.at(i - 1, default=0), sequence.at(i, default=0)]
        )
        dist_routes.append(
            model.sum(model.range(1, c), dist_lambda)
            + model.iif(
                c > 0,
                dist_depot_array[sequence.at(0, default=0)] + dist_depot_array[sequence.at(c - 1, default=0)],
                0,
            )
        )

        # Recursive end-time array.
        bs = breaks_start_times[k]

        def end_lambda(i, prev, k=k, sequence=sequence, bs=bs):
            customer = sequence.at(i, default=0)
            previous_customer = sequence.at(i - 1, default=0)
            travel_dur = model.iif(
                i == 0,
                dist_depot_array[customer],
                dist_matrix_array[previous_customer, customer],
            )
            travel_end = prev + travel_dur
            # Apply breaks during travel.
            end_with_breaks = travel_end
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(prev <= bs_p, end_with_breaks > bs_p)
                end_with_breaks = model.iif(in_break, end_with_breaks + BREAKDURATION, end_with_breaks)
            # Apply waiting + service.
            next_start = model.max(end_with_breaks, earliest_array[customer])
            service_end = next_start + service_time_array[customer]
            # Apply breaks during service.
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(end_with_breaks <= bs_p, service_end > bs_p)
                service_end = model.iif(
                    in_break,
                    model.max(bs_p + BREAKDURATION, earliest_array[customer]) + service_time_array[customer],
                    service_end,
                )
            return service_end

        end_time_k = model.array(model.range(0, c), model.lambda_function(end_lambda), 0)
        end_times.append(end_time_k)

    # Lateness terms.
    total_lateness_terms = []
    for k in range(nb_trucks):
        sequence = customers_sequences[k]
        c = model.count(sequence)
        end_time_k = end_times[k]
        bs = breaks_start_times[k]

        def home_lateness(prev, k=k, sequence=sequence, c=c, end_time_k=end_time_k, bs=bs):
            last_end = end_time_k.at(c - 1, default=0)
            last_customer = sequence.at(c - 1, default=0)
            return_home = last_end + dist_depot_array[last_customer]
            for p in range(nb_breaks):
                bs_p = model.at(bs, p)
                in_break = model.and_(last_end <= bs_p, return_home > bs_p)
                return_home = model.iif(in_break, return_home + BREAKDURATION, return_home)
            return model.max(0, return_home - max_horizon)

        home_term = model.iif(trucks_used[k], home_lateness(0), 0)

        def visit_lateness(i, k=k, end_time_k=end_time_k, sequence=sequence):
            customer = sequence.at(i, default=0)
            return model.max(0, end_time_k.at(i, default=0) - latest_array[customer])

        visit_term = model.sum(model.range(0, c), model.lambda_function(visit_lateness))
        total_lateness_terms.append(home_term + visit_term)

    total_lateness = model.sum(*total_lateness_terms)
    nb_trucks_used = model.sum(*trucks_used)
    total_distance = model.round(100 * model.sum(*dist_routes)) / 100

    model.minimize(total_lateness, name="total_lateness")
    model.minimize(nb_trucks_used, name="nb_trucks_used")
    model.minimize(total_distance, name="total_distance")

    return model, customers_sequences, total_lateness, nb_trucks_used, total_distance


def main(instance_file, output_file=None, time_limit=20):
    data = read_instance(instance_file)
    print(
        f"customers={data['nb_customers']} trucks={data['nb_trucks']} "
        f"capacity={data['truck_capacity']} horizon={data['max_horizon']}"
    )
    model, customers_sequences, total_lateness, nb_trucks_used, total_distance = build_cvrptwrb_model(data)
    solution = solve(model, time_limit_s=float(time_limit))
    result_values = solution.values(
        {
            "total_lateness": total_lateness,
            "trucks_used": nb_trucks_used,
            "total_distance": total_distance,
            **{f"truck_{k}": sequence for k, sequence in enumerate(customers_sequences)},
        }
    )

    lines = [
        f"Total lateness = {result_values['total_lateness']}; "
        f"Trucks used = {result_values['trucks_used']}; "
        f"Total distance = {result_values['total_distance']}; "
        f"Status = {solution.status.value}"
    ]
    for truck in range(data["nb_trucks"]):
        sequence = result_values[f"truck_{truck}"]
        if sequence:
            customers = " ".join(str(customer + 1) for customer in sequence)
            lines.append(f"Truck {truck + 1}: {customers}")

    result_text = "\n".join(lines)
    print(result_text)
    if output_file is not None:
        Path(output_file).write_text(result_text + "\n", encoding="utf-8")
    return solution

## 运行实例

Notebook 直接调用 `main` 并显式传入实例路径。以下三个代码格相互独立,可以按需要单独运行;调整 `time_limit` 可以控制每个实例的求解时间。


In [2]:
INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)

Instances: /Users/dongbox/work/optagent/examples/examples/hexaly/capacitated_vehicle_routing_problem_with_time_windows_and_regular_breaks/instances


In [3]:
solution_c101_25 = main(
    INSTANCE_DIR / "C101.25.txt",
    time_limit=10,
)

Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0


customers=25 trucks=25 capacity=200 horizon=1236


Solve summary:
  status: FEASIBLE
  objective: 1916.604148556141
  improvements: initial=3 search=0
  evaluated: 0
  wall_time: 10.0621s
  termination: wall_time_exhausted


Total lateness = 1916.604148556141; Trucks used = 16; Total distance = 904.19; Status = feasible
Truck 1: 22 19
Truck 3: 11
Truck 4: 20 25
Truck 5: 16 24 23
Truck 6: 10 7
Truck 7: 18
Truck 9: 13
Truck 10: 21
Truck 11: 2
Truck 14: 9 1 12
Truck 16: 15
Truck 18: 3 17
Truck 19: 5
Truck 22: 6
Truck 24: 14
Truck 25: 8 4


In [4]:
solution_c101_50 = main(
    INSTANCE_DIR / "C101.50.txt",
    time_limit=10,
)

Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0


customers=50 trucks=25 capacity=200 horizon=1236


Solve summary:
  status: FEASIBLE
  objective: 8948.922009888835
  improvements: initial=4 search=0
  evaluated: 0
  wall_time: 10.3473s
  termination: wall_time_exhausted


Total lateness = 8948.922009888835; Trucks used = 22; Total distance = 1664.77; Status = feasible
Truck 1: 10 35 29 47
Truck 2: 31
Truck 3: 18 49 14
Truck 4: 41
Truck 5: 36 39 30 37 8
Truck 6: 23 20
Truck 8: 12 34
Truck 9: 32 43
Truck 10: 9 5 19
Truck 11: 17 13
Truck 12: 33 42
Truck 13: 44
Truck 14: 3 2 6
Truck 15: 28 24 1
Truck 16: 48
Truck 17: 25 21
Truck 18: 46 40
Truck 19: 45 50
Truck 20: 15
Truck 22: 22 27 11
Truck 23: 7 26
Truck 25: 4 38 16


In [5]:
solution_c101_100 = main(
    INSTANCE_DIR / "C101.100.txt",
    time_limit=10,
)

Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0


customers=100 trucks=25 capacity=200 horizon=1236


Solve summary:
  status: FEASIBLE
  objective: 37415.74131748287
  improvements: initial=1 search=0
  evaluated: 0
  wall_time: 10.0564s
  termination: wall_time_exhausted


Total lateness = 37415.74131748287; Trucks used = 25; Total distance = 4452.26; Status = feasible
Truck 1: 95 33 32
Truck 2: 94 7 61 10 85
Truck 3: 34 18
Truck 4: 80 91 41 69
Truck 5: 89 30
Truck 6: 40 44
Truck 7: 76 14 13 48
Truck 8: 63 57 9 66
Truck 9: 79 19 23 1 62 2 67
Truck 10: 87 20 35 49 70 39 50
Truck 11: 90 8 72 78 81
Truck 12: 12 54 52 24
Truck 13: 96 60
Truck 14: 100 74
Truck 15: 6 43 68 29
Truck 16: 97 21 25
Truck 17: 38 64 82
Truck 18: 15 42 93 11
Truck 19: 5 86 28 22
Truck 20: 31 83 51
Truck 21: 16 46 77 45 17 4 58 65 36
Truck 22: 92 98 73 53
Truck 23: 59 55
Truck 24: 84 47 3 56 71
Truck 25: 75 99 37 26 27 88
